# Exercise 6: An AI Inference Pipeline for Wildfire Mapping

The earlier exercises each trained or fine tuned a model on a fixed collection of small image patches. In practice, once a model exists, it needs to be applied to an entire area of interest, which is usually much larger than the patches used for training and does not come pre packaged as a tidy dataset. This exercise builds a small pipeline that goes from a geographic extent all the way to a finished map, saved as a GeoTIFF that can be opened in any GIS software.

The pipeline reuses the U-Net architecture and the six Sentinel 2 bands from Exercise 2, so a model trained there could be plugged directly into this pipeline.

## Pipeline steps

1. Download a cloud free Sentinel 2 composite for a chosen area and period, using openEO, following the same approach as Exercise 1
2. Split the downloaded scene into fixed size tiles, since the model was trained on small patches rather than whole regions
3. Run the trained model on every tile
4. Stitch the tile predictions back into a single map covering the full area
5. Save the result as a GeoTIFF, with the same coordinate reference system and geographic transform as the input imagery

## Code organisation

Unlike the previous exercises, most of the logic for this exercise lives in a small Python package rather than directly in the notebook. The `pipeline` folder next to this notebook contains one file per step:

- `download.py` connects to openEO and produces the input composite
- `tiling.py` reads a raster and splits it into tiles
- `model.py` defines the U-Net architecture and loads its weights
- `inference.py` runs the model over a list of tiles
- `mosaic.py` stitches predictions back together and writes the output GeoTIFF
- `pipeline.py` combines all of the above into a single function, `run_pipeline`

This notebook first walks through each step individually, so every part of the process is visible, and then shows how the same result is obtained with a single function call.

In [ ]:
!pip install torch rasterio openeo matplotlib numpy -q

## Getting the `pipeline` package when running in Colab

This notebook imports its pipeline steps from the `pipeline` folder that sits next to it in the repository. Opening the notebook directly in Google Colab only fetches the notebook itself, not the rest of the repository, so the folder needs to be fetched separately before it can be imported. The cell below detects whether the notebook is running in Colab and, if so, downloads the `pipeline` folder from the course GitHub repository. Running locally with the repository already cloned does not require this step, and the cell leaves an existing `pipeline` folder untouched.

In [ ]:
import os
import shutil
import subprocess
import sys

if "google.colab" in sys.modules and not os.path.isdir("pipeline"):
    repo_dir = "GEOAI4NHM_repo"
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/ashokdahal/GEOAI4NHM.git", repo_dir],
        check=True,
    )
    shutil.copytree(os.path.join(repo_dir, "pipeline"), "pipeline")
    shutil.rmtree(repo_dir)
    print("Downloaded the pipeline package from GitHub.")
else:
    print("pipeline package already available, nothing to download.")

In [ ]:
import sys
sys.path.append(".")

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

from pipeline import (
    download_composite,
    read_raster_as_array,
    pad_to_multiple,
    split_into_tiles,
    load_model,
    run_inference_on_tiles,
    stitch_tiles,
    save_prediction_as_geotiff,
    run_pipeline,
)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Defining the area and period of interest

The pipeline needs a geographic extent and a period over which to build a cloud free composite, in exactly the same format used in Exercise 1. The example below uses an area in Northern California that has previously been affected by wildfire, over a period covering both pre fire and post fire months, so that the resulting composite has a reasonable chance of showing burned vegetation. Replace these values with any other area and period you are interested in mapping.

In [ ]:
extent = {"west": -121.65, "south": 39.72, "east": -121.50, "north": 39.82}
temporal_extent = ["2018-10-01", "2019-01-31"]

output_dir = "data/inference_run"

## Step 1: Downloading the input composite

`download_composite` follows the same recipe introduced in Exercise 1, masking clouds using the Scene Classification Layer and reducing the resulting time series to a single median composite, using the six bands the U-Net expects. Running this step can take a few minutes, since it triggers processing on the openEO backend.

In [ ]:
scene_path = download_composite(extent, temporal_extent, output_dir=f"{output_dir}/scene")
print(f"Downloaded composite: {scene_path}")

## Step 2: Reading and tiling the scene

The downloaded composite is read into memory as an array with shape (bands, height, width), together with a rasterio profile that records its coordinate reference system and geographic transform. Since the model expects 256 by 256 pixel inputs, the scene is padded so its dimensions are an exact multiple of the tile size, then split into a grid of tiles. The reflectance values are also scaled to approximately the 0 to 1 range, matching the normalisation used during training in Exercise 2.

In [ ]:
TILE_SIZE = 256

scene_array, profile = read_raster_as_array(scene_path)
scene_array = np.clip(scene_array / 10000.0, 0.0, 1.0)

padded_array, original_shape = pad_to_multiple(scene_array, TILE_SIZE)
tiles, offsets = split_into_tiles(padded_array, TILE_SIZE)

print(f"Original scene shape : {scene_array.shape}")
print(f"Padded scene shape   : {padded_array.shape}")
print(f"Number of tiles      : {len(tiles)}")

In [ ]:
def false_colour_composite(image):
    """Build a Shortwave Infrared, Near Infrared, Red composite for display, scaled to 0 to 1."""
    composite = image[[4, 3, 2], :, :]
    composite = np.moveaxis(composite, 0, -1)
    return np.clip(composite * 2.5, 0, 1)


example_tile_index = len(tiles) // 2
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(false_colour_composite(tiles[example_tile_index]))
ax.set_title(f"Example tile at offset {offsets[example_tile_index]}")
ax.axis("off")
plt.show()

## Step 3: Running the model on every tile

`load_model` builds the same U-Net architecture used in Exercise 2. Passing a `checkpoint_path` loads weights saved from that exercise's training run. If no checkpoint is available yet, the model runs with random weights, which is enough to confirm that the pipeline works end to end, though the resulting map will not be meaningful until real trained weights are supplied.

To use your own trained weights, first save them at the end of Exercise 2 with `torch.save(model.state_dict(), "unet_burn_scars.pt")`, then point `checkpoint_path` below to that file.

In [ ]:
checkpoint_path = None  # set to a path such as "unet_burn_scars.pt" once you have trained weights

model = load_model(checkpoint_path=checkpoint_path, device=device)
tile_predictions = run_inference_on_tiles(model, tiles, device)

print(f"Produced {len(tile_predictions)} tile predictions, each of shape {tile_predictions[0].shape}")

## Step 4: Stitching predictions into a single map

Because the tiles do not overlap, their predictions can be placed directly back into a full sized array at the pixel offsets recorded during tiling. The padding added earlier is then cropped away, restoring the original scene dimensions.

In [ ]:
padded_mosaic = stitch_tiles(tile_predictions, offsets, TILE_SIZE, padded_array.shape[1:])
mosaic = padded_mosaic[:original_shape[0], :original_shape[1]]

print(f"Final map shape: {mosaic.shape}")

## Step 5: Saving the result as a GeoTIFF

The predicted map is saved using the coordinate reference system and geographic transform of the input composite, so the output lines up correctly with the source imagery and can be opened directly in GIS software such as QGIS.

In [ ]:
output_path = f"{output_dir}/burn_scar_map.tif"
save_prediction_as_geotiff(mosaic, profile, output_path)
print(f"Saved prediction map to {output_path}")

## Viewing the result

The cell below displays the false colour input composite next to the predicted map, so the two can be compared directly.

In [ ]:
with rasterio.open(scene_path) as src:
    full_scene = src.read().astype(np.float32)

with rasterio.open(output_path) as src:
    prediction_map = src.read(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(false_colour_composite(full_scene / 10000.0))
axes[0].set_title("Input composite (SWIR1, NIR, Red)")
axes[1].imshow(prediction_map, cmap="RdYlGn_r", vmin=0, vmax=1)
axes[1].set_title("Predicted burn scar map")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Running everything with one function call

Now that every step has been inspected individually, the same result can be produced with a single call to `run_pipeline`, which is convenient once the pipeline has been checked and is ready to be applied to a new area or period without further inspection at each step.

In [ ]:
final_output_path = run_pipeline(
    extent=extent,
    temporal_extent=temporal_extent,
    output_dir="data/inference_run_direct",
    checkpoint_path=checkpoint_path,
    tile_size=TILE_SIZE,
    device=device,
)
print(f"Pipeline finished, output saved to {final_output_path}")

## Summary

This exercise connected the two ends of a typical Earth observation modelling project: downloading imagery for an arbitrary area of interest through openEO, in the same way as Exercise 1, and applying a trained segmentation model, in the same way as Exercise 2, at a scale much larger than the small patches used during training.

Splitting a large scene into tiles, running a model on each tile, and stitching the results back together is a pattern that applies well beyond this specific example. The same three steps, tiling, inference, and stitching, would apply equally to the SegFormer model from Exercise 3 or the TerraMind model from Exercise 4, by replacing the contents of `model.py` while keeping the rest of the pipeline unchanged.